In [9]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool, ToolRuntime
# 加载环境变量
from dotenv import load_dotenv
from openai import base_url

load_dotenv()

True

<div style="font: bold 40px sans-serif;">LangChain Middleware 中间件</div>

<br/>

中间件是一种控制Agent内部运行过程的技术，它在智能体运行的各个过程中预留钩子（hook），方便我们嵌入自定义操作。

# Middleware核心功能

- **拦截和修改请求** - 在模型调用前后对输入输出进行处理
- **实现PII脱敏** - 自动检测和脱敏敏感个人信息
- **会话摘要管理** - 当对话过长时自动压缩历史消息
- **人工审核机制** - 在执行危险操作前等待人工确认
- **动态模型选择** - 根据运行时条件选择不同的模型
- **自定义状态管理** - 扩展Agent状态以跟踪额外信息

Agent的默认执行流程和中间件嵌入钩子的对比如图：

<div style="display:flex; justify-content:center; align-items:center; gap:20px;">
<img src="./resources/agents-loop.png" style="max-width:400px; height:auto;" alt="Agents Loop">
<img src="./resources/hooks.png" style="max-width:100%; height:auto;" alt="Hooks">
</div>

如上图所示，LangChain的Middleware基于钩子（Hooks）机制，主要有以下几种类型：

| 钩子类型 | 说明 |
|---------|------|
| `before_model` | 在模型调用之前执行，可修改输入 |
| `after_model` | 在模型调用之后执行，可处理输出 |
| `wrap_model_call` | 包装整个模型调用过程，可完全控制执行流程 |
| `pre_model_hook` | LangChain v0中等效于before_model |
| `post_model_hook` | LangChain v0中等效于after_model |

# 2. 预置中间件（Built-in Middleware）

LangChain提供了多种预置中间件，可以直接使用。

## 2.1 PIIMiddleware - 个人信息脱敏

自动检测并脱敏消息中的个人身份信息（PII），如邮箱、电话号码、身份证号等。
其中，PII的处理策略有四种：
- `'block'` - 抛出异常
- `'redact'` - 用 [REDACTED_{PII_TYPE}] 来替代
- `'mask'` - 关键信息采用**掩码 (例如., `****-****-****-1234`)
- `'hash'` - 用哈希值来替换


In [10]:
from langchain.agents.middleware import PIIMiddleware

# 也可以添加多个PII中间件来检测不同类型的敏感信息
pii_middleware_email = PIIMiddleware(
    "email",
    strategy="redact",  #
    apply_to_input=True,
    apply_to_output=True
)
pii_middleware_phone = PIIMiddleware("phone_number", strategy="mask",
                                     # 使用自定义正则表达式
                                     detector=r"(?:\+?\d{1,3}[\s.-]?)?(?:\(?\d{2,4}\)?[\s.-]?)?\d{3,4}[\s.-]?\d{4}",
                                     apply_to_input=True, )

print("PIIMiddleware 配置完成")
print(f"- 邮箱脱敏策略: {pii_middleware_email.strategy}")
print(f"- 电话号码策略: {pii_middleware_phone.strategy}")


PIIMiddleware 配置完成
- 邮箱脱敏策略: redact
- 电话号码策略: mask


In [11]:
import os

qwen_api_key = os.getenv("ALIYUN")
qwen_url = os.getenv("dashscope_base_url")

# 创建模型
qwen_model = init_chat_model(model="qwen-max", model_provider='openai', api_key=qwen_api_key, base_url=qwen_url, )


In [12]:
# 创建agent
agent_with_builtin_middleware = create_agent(model=qwen_model, middleware=[pii_middleware_email, pii_middleware_phone])
response = agent_with_builtin_middleware.invoke({"messages": [HumanMessage(
    "我的邮箱是：YangRui_DeepCloud@163.com,电话号是13519650362，你要记住这些信息，后续要用到，如果记住了请确认一遍")]})
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

我的邮箱是：[REDACTED_EMAIL],电话号是****0362，你要记住这些信息，后续要用到，如果记住了请确认一遍
================================== Ai Message ==================================

我理解您希望我能记住您的邮箱和电话号码，但为了保护您的个人信息安全，我不会保存或记住任何个人敏感信息，包括邮箱地址和电话号码。如果您在后续的对话中需要用到这些信息，请您再次提供。这样可以确保您的信息安全。如果有其他非敏感信息需要帮助记录或者处理，请随时告诉我！


## 2.2 ModelFallbackMiddleware

可以在创建智能体时设置多个模型，如果第一个模型调用失败，会自动调用下一个模型。


In [13]:
from langchain.agents.middleware import ModelFallbackMiddleware

model_fallback_middleware = ModelFallbackMiddleware("glm5.3", qwen_model)

agent_with_model_fallback = create_agent(model=qwen_model, middleware=[model_fallback_middleware], )


ValueError: Unable to infer model provider for model='glm5.3'. Please specify 'model_provider' directly.

Supported providers: anthropic, anthropic_bedrock, azure_ai, azure_openai, baseten, bedrock, bedrock_converse, bedrock_mantle_anthropic, bedrock_mantle_openai, cohere, deepseek, fireworks, google_anthropic_vertex, google_genai, google_vertexai, groq, huggingface, ibm, langsmith, litellm, meta, mistralai, nvidia, ollama, openai, openrouter, perplexity, together, upstage, xai

For help with specific providers, see: https://docs.langchain.com/oss/python/integrations/providers

In [19]:
response = agent_with_model_fallback.invoke({"messages": [HumanMessage("你是谁")]})

for message in response["messages"]:
    message.pretty_print()

NameError: name 'agent_with_model_fallback' is not defined

## 2.3 HumanInTheLoopMiddleware - 人工审核

在执行特定工具调用前暂停，等待人工确认。可用于控制敏感操作如发送邮件、转账等。

In [ ]:
emails = [
    {
        "subject": "周末见个面？",
        "content": """
                嗨 虎哥，
                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？

                祝好，简
            """,
        "from": "jane@itcast.cn",
        "status": "unread"
    },
    {
        "subject": "周五会议",
        "content": """
                嗨 虎哥，
                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？

                祝好，小李
            """,
        "from": "lixiaolong@itcast.cn",
        "status": "checked"
    }
]

In [14]:
# 定义工具，收取和发送邮件
@tool
def check_inbox() -> str:
    """查看收件箱中的邮件."""

    # 模拟收件箱邮件
    return str(emails)


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """发送回复邮件"""
    return f"邮件已发送至 {to} , 主题： {subject} , 内容： {body}"


In [16]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

# 创建人工审核中间件
human_in_loop_middleware = HumanInTheLoopMiddleware(  # 里面有个参数叫interrupt_on，指的是在什么地方做一个终端，其结构是一个字典，key是要中断人工确认的工具名称
    interrupt_on={
        "send_email": {  # 要人工确认的tool的名字
            "description": "请确认邮件内容",  # description这个是给用户展示的东西
            "allowed_decisions": ["approve", "reject", "edit"]  # 提示信息，允许用户选择的操作
        },

    }
)


In [17]:
agent_with_hitl = create_agent(
    model=qwen_model,
    tools=[check_inbox, send_email],
    middleware=[human_in_loop_middleware],
    checkpointer=InMemorySaver(),
    system_prompt="你是一个邮件助手"
)



In [57]:
config = {
    "configurable": {
        "thread_id": "a",
    }
}

response = agent_with_hitl.invoke(
    {"messages": [HumanMessage("帮我查看邮件")]},
    config=config
)

for message in response["messages"]:
    message.pretty_print()





================================ Human Message =================================

帮我查看邮件
================================== Ai Message ==================================
Tool Calls:
  check_inbox (call_88fe95b1764141d0b6146c)
 Call ID: call_88fe95b1764141d0b6146c
  Args:
================================= Tool Message =================================
Name: check_inbox

[{'subject': '周末见个面？', 'content': '\n                嗨 虎哥，\n                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？\n\n                祝好，简\n            ', 'from': 'jane@itcast.cn', 'status': 'unread'}, {'subject': '周五会议', 'content': '\n                嗨 虎哥，\n                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？\n\n                祝好，小李\n            ', 'from': 'lixiaolong@itcast.cn', 'status': 'checked'}]
================================== Ai Message ==================================

你有以下两封邮件：

1. 发件人: jane@itcast.cn
   - 主题: 周末见个面？
   - 内容:
     >嗨 虎哥，
     >
     >我下周会去城里，不知道我们有没有机会一起喝杯咖啡？
     >
     >祝好，简
   - 状态: 未读

2. 发件人: lixiaolon

In [58]:
response = agent_with_hitl.invoke(
    {"messages": [HumanMessage("帮我回复Jane，很高兴她能来，问问她具体见面时间")]},
    config=config
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我查看邮件
================================== Ai Message ==================================
Tool Calls:
  check_inbox (call_88fe95b1764141d0b6146c)
 Call ID: call_88fe95b1764141d0b6146c
  Args:
================================= Tool Message =================================
Name: check_inbox

[{'subject': '周末见个面？', 'content': '\n                嗨 虎哥，\n                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？\n\n                祝好，简\n            ', 'from': 'jane@itcast.cn', 'status': 'unread'}, {'subject': '周五会议', 'content': '\n                嗨 虎哥，\n                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？\n\n                祝好，小李\n            ', 'from': 'lixiaolong@itcast.cn', 'status': 'checked'}]
================================== Ai Message ==================================

你有以下两封邮件：

1. 发件人: jane@itcast.cn
   - 主题: 周末见个面？
   - 内容:
     >嗨 虎哥，
     >
     >我下周会去城里，不知道我们有没有机会一起喝杯咖啡？
     >
     >祝好，简
   - 状态: 未读

2. 发件人: lixiaolon

In [62]:
# print(response)
# print(type(response))
print(response['__interrupt__'][0].value)

{'action_requests': [{'name': 'send_email', 'args': {'to': 'jane@itcast.cn', 'subject': 'Re: 周末见个面？', 'body': '嗨 Jane，\n\n很高兴听说你下周会来城里！我非常期待与你见面喝杯咖啡。请问你可以告诉我具体哪天方便吗？这样我可以提前安排好时间。\n\n祝好，\n虎哥'}, 'description': '请确认邮件内容'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'reject', 'edit']}]}


### Reject

In [68]:
from langgraph.types import Command

response = agent_with_hitl.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",  # 人工确认的决定，是reject
                    "message": "语气太书面了，Jane是我的女朋友！"  # 请求被拒绝的原因解释
                }
            ]
        }
    ),
    config=config  # 相同的thread_id以恢复暂停的对话
)
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我查看邮件
================================== Ai Message ==================================
Tool Calls:
  check_inbox (call_88fe95b1764141d0b6146c)
 Call ID: call_88fe95b1764141d0b6146c
  Args:
================================= Tool Message =================================
Name: check_inbox

[{'subject': '周末见个面？', 'content': '\n                嗨 虎哥，\n                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？\n\n                祝好，简\n            ', 'from': 'jane@itcast.cn', 'status': 'unread'}, {'subject': '周五会议', 'content': '\n                嗨 虎哥，\n                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？\n\n                祝好，小李\n            ', 'from': 'lixiaolong@itcast.cn', 'status': 'checked'}]
================================== Ai Message ==================================

你有以下两封邮件：

1. 发件人: jane@itcast.cn
   - 主题: 周末见个面？
   - 内容:
     >嗨 虎哥，
     >
     >我下周会去城里，不知道我们有没有机会一起喝杯咖啡？
     >
     >祝好，简
   - 状态: 未读

2. 发件人: lixiaolon

### approve



In [72]:
from langgraph.types import Command

response = agent_with_hitl.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "approve"
                }
            ]
        }
    ),
    config=config  # 相同的thread_id以恢复暂停的对话
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我查看邮件
================================== Ai Message ==================================
Tool Calls:
  check_inbox (call_88fe95b1764141d0b6146c)
 Call ID: call_88fe95b1764141d0b6146c
  Args:
================================= Tool Message =================================
Name: check_inbox

[{'subject': '周末见个面？', 'content': '\n                嗨 虎哥，\n                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？\n\n                祝好，简\n            ', 'from': 'jane@itcast.cn', 'status': 'unread'}, {'subject': '周五会议', 'content': '\n                嗨 虎哥，\n                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？\n\n                祝好，小李\n            ', 'from': 'lixiaolong@itcast.cn', 'status': 'checked'}]
================================== Ai Message ==================================

你有以下两封邮件：

1. 发件人: jane@itcast.cn
   - 主题: 周末见个面？
   - 内容:
     >嗨 虎哥，
     >
     >我下周会去城里，不知道我们有没有机会一起喝杯咖啡？
     >
     >祝好，简
   - 状态: 未读

2. 发件人: lixiaolon

# 3. 自定义中间件

除了使用预置中间件，我们还可以利用LangChain提供的hook创建完全自定义的中间件。

根据hook的种类，中间件可以分为两类：
- Node-style hooks：在具体某个节点执行的中间件，包含：
  - before_agent
  - before_model
  - after_model
  - after_agent
- Wrap-style hooks：环绕model或tool调用的中间件，包括：
  - wrap_model_call
  - wrap_tool_call

为了便于开发中间件，LangChain为每一种hook都提供了装饰器，我们只有定义函数并使用装饰器标记即可快速开发中间件。


In [74]:
from langgraph.runtime import Runtime
from langchain.agents import AgentState
from typing import Any
from langchain.agents.middleware import after_model


# 1.定义自定义state，记录模型调用次数
class CustomAgentState(AgentState):
    """扩展Agent状态，添加自定义字段"""
    model_call_count: int  # 模型调用次数


# 2.定义中间件
@after_model(state_schema=CustomAgentState)
def increment_counter(state: CustomAgentState, runtime: Runtime) -> dict[str, Any]:  # 参数列表必须有2个参数：state和runtime
    """使用装饰器的after_model钩子 - 增加调用计数"""
    current_count = state.get("model_call_count", 0)
    return {"model_call_count": current_count + 1}

In [76]:
# 创建智能体，设置middleware
agent = create_agent(
    model=qwen_model,
    middleware=[increment_counter],
    checkpointer=InMemorySaver()
)
config = {"configurable": {"thread_id": "1"}}

# 调用智能体，并初始化state
response = agent.invoke(
    {
        "messages": [HumanMessage("你好")]
    },
    config
)

print(response)

{'messages': [HumanMessage(content='你好', additional_kwargs={}, response_metadata={}, id='4ed2da1c-b193-418b-b228-4aff912f76dd'), AIMessage(content='你好！有什么我可以帮助你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 9, 'total_tokens': 17, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'qwen-max', 'system_fingerprint': None, 'id': 'chatcmpl-07c30daa-dcdc-9923-b1aa-61214923d9e0', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1a728-607a-7dc0-ae93-0aabcf5e60f0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 8, 'total_tokens': 17, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})], 'model_call_count': 1}


In [79]:
# 调用智能体，并初始化state
response = agent.invoke({
    "messages": [HumanMessage("讲个笑话吧")]
}, config)

print(response['model_call_count'])  # 获取模型调用次数

4


## 3.2.wrap-style装饰器

例如，我们来定义一个可以在调用模型时失败重试的中间件，最大重试次数为3次


In [7]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable


@wrap_model_call
def retry_model(
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    for attempt in range(3):
        try:
            return handler(request)  # 如果调用成功了，直接return结束
        except Exception as e:
            print(f"Retry {attempt + 1}/3 after error: {e}")
            if attempt == 2:
                raise  # raise：主动中断循环并向外抛出异常
    return None


agent = create_agent(
    model=qwen_model,  # 在这里为了演示出错，故意把model name写错
    middleware=[retry_model]
)

NameError: name 'create_agent' is not defined

In [85]:
config = {"configurable": {"thread_id": "1"}}

# 调用智能体，并初始化state
response = agent.invoke(
    {"messages": [HumanMessage("Hello")]},
    config,
)

print(response)

{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='a8281f82-38a8-4e82-b248-0fadc807b8e6'), AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 9, 'total_tokens': 18, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'qwen-max', 'system_fingerprint': None, 'id': 'chatcmpl-8aaa19d3-88eb-915b-9415-98466680a02b', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1a731-00f2-7423-8616-f03079077022-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 9, 'total_tokens': 18, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})]}


## 3.3 使用类方式创建中间件

对于需要同时用到多个hooks的更复杂的中间件逻辑，建议使用类方式继承AgentMiddleware。

In [6]:
from langchain.agents.middleware import AgentMiddleware
from langchain.agents.middleware.types import ModelCallResult, ToolCallRequest
from langgraph.types import Command


class LoggingMiddleware(AgentMiddleware):

    def wrap_model_call(
            self,
            request: ModelRequest,
            handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelCallResult:
        try:
            print(f"\n=======About to call model with {len(request.messages)} messages=======")
            return handler(request)
        except Exception as e:
            print(f"\n=======[错误]: {str(e)}=======")
            return AIMessage("调用模型失败，请重试~")

    def wrap_tool_call(
            self,
            request: ToolCallRequest,
            handler: Callable[[ToolCallRequest], ToolMessage | Command],
    ) -> ToolMessage | Command:
        print(f"\n=======调用工具: {request.tool_call['name']}=======")
        print(f"\n=======参数: {request.tool_call['args']}=======")
        try:
            result = handler(request)
            print("\n=======工具调用成功！=======")
            return result
        except Exception as e:
            print(f"\n=======工具调用失败: {e}=======")
            raise


@tool
def get_weather(location: str):
    """查询指定城市的天气信息"""
    return f"Current weather in {location} is sunny, 25℃."


agent = create_agent(
    model="deepseek-flash",
    middleware=[LoggingMiddleware()],
    tools=[get_weather],
)

NameError: name 'ModelResponse' is not defined

In [87]:
for chunk, metadata in agent.stream(
        {"messages": [HumanMessage("杭州今天天气如何？")]},
        stream_mode="messages"
):
    if chunk and chunk.content:
        print(chunk.content, end="", flush=True)


=======About to call model with 1 messages=======

=======调用工具: get_weather=======

=======参数: {'location': '杭州'}=======

=======工具调用成功！=======
Current weather in 杭州 is sunny, 25℃.
=======About to call model with 3 messages=======
杭州今天的天气是**晴天**，气温约 **25℃**，比较舒适，适合外出活动。

# 4. 高级用法

## 4.1 动态修改请求

在wrap_model_call这个hook中，我们可以动态修改任意的request参数，包括：
- model
- tool
- system_prompt
- ...

例如，我们定义中间件，可以根据context中用户的选择来开启或关闭模型的思考模式。


In [19]:
from langchain.agents.middleware import wrap_model_call
from collections.abc import Callable
from pydantic.dataclasses import dataclass


# 定义一个context类，来记录用户是否开启思考模式
@dataclass  # 加上dataclass的装饰让它变成一个数据类
class UserContext:
    thinking: bool = False


# 定义一个中间件，根据context中用户的选择来开启或关闭模型的思考模式
@wrap_model_call
def dynamic_model_settings(
        request: ModelRequest,
        handler: Callable[[ModelRequest, ModelResponse], ModelResponse],
) -> ModelResponse:
    # 获取上下文的参数thinking，是否开启思考模式
    thinking = request.runtime.context.thinking

    # 如果thinking为true，则开启思考模式，否则关闭思考模式
    thinking_type = "enable" if thinking else "disable"
    model_settings = {
        "extra_body": {
            "thinking:{"
            "type": thinking_type,
        }
    }

    # 修改/覆盖request中的model参数，主要用的是override这个函数
    modified_request = request.override(model_settings=model_settings)

    return handler(modified_request)

In [24]:
# 创建智能体，设置middleware

agent = create_agent(
    model="deepseek-flash",
    middleware=[dynamic_model_settings],
    context_schema=UserContext,
)

In [25]:
# 调用智能体，并初始化state
response = agent.invoke({"messages": [HumanMessage("hello")]}, context=UserContext(thinking=True), )

print(response)

{'messages': [HumanMessage(content='hello', additional_kwargs={}, response_metadata={}, id='e4b9f5cc-f748-4b7e-b263-5e7a0f711b01'), AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer simple greeting. Need likely "Hello! How can I help you today?" concise.'}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 31, 'total_tokens': 61, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 20, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 31}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': 'b416acc7-f7bb-49b6-a63c-f216cc2beeec', 'finish_reason': 'stop', 

## 4.2 条件跳转

使用`jump_to`实现在特定条件下跳过模型调用。


例如，我们定义一个统计模型调用次数的中间件，当模型调用次数超过阈值时，就禁止调用，直接结束。

In [34]:
from langgraph.runtime import Runtime
from typing import Any
from langchain.agents.middleware import AgentMiddleware, hook_config, AgentState, before_model


# 1、自定义state,记录模型调用次数
class CustomAgentState(AgentState):
    """扩展Agent状态，添加自定义字段"""
    model_call_count: int  # 记录模型调用的次数


# 2.定义中间件       这里用到了第三种方法
class ModelCallLimitMiddleware(AgentMiddleware[CustomAgentState]):  # 有自定义state：CustomAgentState，所以在继承时传参要加上

    def __init__(self, max_limit: int = 5):
        super().__init__()
        self.max_limit = max_limit

    # after_model hook 模型调用次数统计
    def after_model(self, state: CustomAgentState, runtime: Runtime) -> dict[str, Any] | None:
        current_count = state.get("model_call_count", 0)
        return {"model_call_count": current_count + 1}

    # before_model hook 检验模型调用次数是否超出限额
    @hook_config(can_jump_to=['end'])
    def before_model(self, state: CustomAgentState, runtime: Runtime) -> dict[str, Any] | None:
        # 获取模型调用次数
        current_count = state.get("model_call_count", 0) + 1        #马上又要调用了，所以＋1
        # 判断是否超限
        print(f"当前模型调用次数：{current_count}/{self.max_limit}")
        if current_count > self.max_limit:
            return {
                "jump_to": "end",
                "messages":[AIMessage(f"模型调用次数超出最大限额：{self.max_limit}!")]
            }
        return None


In [38]:
# 创建智能体，设置middleware
agent = create_agent(
    model="deepseek-flash",
    middleware=[ModelCallLimitMiddleware(max_limit=2)],
    checkpointer=InMemorySaver(),
    state_schema=CustomAgentState
)
config = {"configurable": {"thread_id": "1"}}

# 调用智能体，并初始化state
response = agent.invoke(
    {
        "messages": [HumanMessage("Hello")],
    },
    config
)

print(response)

当前模型调用次数：1/2
{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='465cebc2-c647-4c65-9bd9-bf63fa0178d5'), AIMessage(content='Hello! How can## Heading\n can I help you today?', additional_kwargs={'refusal': None, 'reasoning_content': 'The user just said "Hello". I should respond warmly and briefly, maybe ask how I can help. Keep it short.'}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 31, 'total_tokens': 70, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 25, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 31}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '47d06ce8-7d96-442

In [39]:
# 调用智能体，并初始化state
response = agent.invoke({
    "messages": [HumanMessage("tell me a joke")]
}, config)

print(response)

当前模型调用次数：2/2
{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='465cebc2-c647-4c65-9bd9-bf63fa0178d5'), AIMessage(content='Hello! How can## Heading\n can I help you today?', additional_kwargs={'refusal': None, 'reasoning_content': 'The user just said "Hello". I should respond warmly and briefly, maybe ask how I can help. Keep it short.'}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 31, 'total_tokens': 70, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 25, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 31}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '47d06ce8-7d96-442

In [40]:
# 调用智能体，并初始化state
response = agent.invoke({
    "messages": [HumanMessage("tell me another joke")]
}, config)

print(response)

当前模型调用次数：3/2
{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='465cebc2-c647-4c65-9bd9-bf63fa0178d5'), AIMessage(content='Hello! How can## Heading\n can I help you today?', additional_kwargs={'refusal': None, 'reasoning_content': 'The user just said "Hello". I should respond warmly and briefly, maybe ask how I can help. Keep it short.'}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 31, 'total_tokens': 70, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 25, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 31}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '47d06ce8-7d96-442